<a href="https://colab.research.google.com/github/kaziibtisumwasit/Thesis_Parkinson-s-Disease-Detection-from-Voice-Recording/blob/main/Parkinson%E2%80%99s_Disease_Detection_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys
if 'google.colab' in sys.modules:
    !pip install --upgrade pip
    !pip install praat-parselmouth opensmile

# ===================== MOUNT DRIVE =====================
from google.colab import drive
drive.mount('/content/drive')

# ===================== IMPORTS =====================
import os
import numpy as np
import pandas as pd
import librosa
import parselmouth
import opensmile
from scipy.stats import skew, kurtosis
from parselmouth.praat import call
import warnings
warnings.filterwarnings("ignore")

# ===================== PROJECT PATHS =====================
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings"
DATA_ROOT = os.path.join(PROJECT_DIR, "raw_data", "data")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "parkinsons_audio_features_combined.csv")

# ===================== HELPER: STATISTICAL SUMMARY =====================
def summarize_feature(arr, prefix):
    stats = {}
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    arr = arr[arr > 0] if np.any(arr > 0) else arr

    stats[f"{prefix}_mean"] = np.mean(arr) if arr.size else 0
    stats[f"{prefix}_std"] = np.std(arr) if arr.size else 0
    stats[f"{prefix}_min"] = np.min(arr) if arr.size else 0
    stats[f"{prefix}_max"] = np.max(arr) if arr.size else 0
    stats[f"{prefix}_range"] = stats[f"{prefix}_max"] - stats[f"{prefix}_min"]
    stats[f"{prefix}_skew"] = skew(arr) if arr.size > 2 else 0
    stats[f"{prefix}_kurtosis"] = kurtosis(arr) if arr.size > 3 else 0
    return stats

# ===================== RHYTHM FEATURES =====================
def calculate_rhythm_features(y, sr):
    features = {}
    frame_length, hop_length = 2048, 512
    RMS_THR, MIN_DUR = 0.25, 0.05

    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    rms = rms / np.max(rms) if np.max(rms) > 1e-6 else np.zeros_like(rms)
    voiced = rms > RMS_THR
    frame_dur = hop_length / sr

    durations, start, active = [], 0, False
    for i in range(len(voiced)):
        if voiced[i] and not active:
            start, active = i, True
        if (not voiced[i] or i == len(voiced) - 1) and active:
            dur = (i - start) * frame_dur
            if dur >= MIN_DUR:
                durations.append(dur)
            active = False

    V = np.array(durations)
    total_time = len(y) / sr

    features["VarcoV"] = np.std(V) / np.mean(V) if V.size and np.mean(V) != 0 else 0
    features["PercentV"] = np.sum(V) / total_time if V.size else 0
    features["nPVI"] = (
        (np.sum(np.abs(np.diff(V))) / np.sum(V[:-1] + V[1:])) *
        (len(V) / (len(V) - 1)) * 100
        if V.size > 1 and np.sum(V[:-1] + V[1:]) != 0 else 0
    )
    return features

# ===================== MAIN FEATURE EXTRACTION =====================
def extract_all_features(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    results = {}

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch()
    harmonicity = snd.to_harmonicity()
    point = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    # -------- JITTER --------
    jp = (0, 0, 0.0001, 0.02, 1.3)
    results["Jitter_local"] = call(point, "Get jitter (local)", *jp)
    results["Jitter_local_abs"] = call(point, "Get jitter (local, absolute)", *jp)
    results["Jitter_rap"] = call(point, "Get jitter (rap)", *jp)
    results["Jitter_ppq5"] = call(point, "Get jitter (ppq5)", *jp)
    results["Jitter_ddp"] = call(point, "Get jitter (ddp)", *jp)

    # -------- SHIMMER --------
    sp = (0, 0, 0.0001, 0.02, 1.3, 1.6)
    results["Shimmer_local"] = call([snd, point], "Get shimmer (local)", *sp)
    results["Shimmer_local_dB"] = call([snd, point], "Get shimmer (local_dB)", *sp)
    results["Shimmer_apq3"] = call([snd, point], "Get shimmer (apq3)", *sp)
    results["Shimmer_apq5"] = call([snd, point], "Get shimmer (apq5)", *sp)
    results["Shimmer_apq11"] = call([snd, point], "Get shimmer (apq11)", *sp)
    results["Shimmer_dda"] = call([snd, point], "Get shimmer (dda)", *sp)

    # -------- HNR --------
    h_vals = harmonicity.values[harmonicity.values > -200]
    results.update(summarize_feature(h_vals, "HNR"))

    # -------- F0 --------
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    results.update(summarize_feature(f0, "F0"))
    results["F0_slope"] = np.polyfit(np.arange(len(f0)), f0, 1)[0] if len(f0) > 1 else 0

    # -------- FORMANTS --------
    formant = snd.to_formant_burg()
    duration = snd.get_total_duration()
    t_vals = np.linspace(0, duration, max(int(duration * 100), 1))

    for i in range(1, 4):
        vals = np.array([formant.get_value_at_time(i, t) for t in t_vals])
        vals = vals[np.isfinite(vals) & (vals > 0)]
        results.update(summarize_feature(vals, f"F{i}"))

    # -------- MFCC + DELTAS --------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for name, arr in [
        ("MFCC", mfcc),
        ("DeltaMFCC", librosa.feature.delta(mfcc)),
        ("Delta2MFCC", librosa.feature.delta(mfcc, order=2)),
        ("Delta3MFCC", librosa.feature.delta(mfcc, order=3)),
    ]:
        for i in range(13):
            results.update(summarize_feature(arr[i], f"{name}{i+1}"))

    # -------- SPECTRAL FEATURES --------
    spectral = {
        "ZCR": librosa.feature.zero_crossing_rate(y)[0],
        "Energy": librosa.feature.rms(y=y)[0],
        "Centroid": librosa.feature.spectral_centroid(y=y, sr=sr)[0],
        "Rolloff": librosa.feature.spectral_rolloff(y=y, sr=sr)[0],
        "Bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr)[0],
        "Flatness": librosa.feature.spectral_flatness(y=y)[0],
    }

    for k, v in spectral.items():
        results.update(summarize_feature(v, k))
        results.update(summarize_feature(librosa.feature.delta(v), f"Delta{k}"))
        results.update(summarize_feature(librosa.feature.delta(v, order=2), f"Delta2{k}"))

    # -------- SPECTRAL CONTRAST --------
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        results.update(summarize_feature(contrast[i], f"SpecContrast{i+1}"))

    # -------- CHROMA --------
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        results.update(summarize_feature(chroma[i], f"Chroma{i+1}"))

    # -------- MEL BANDS --------
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=25)
    for i, val in enumerate(np.mean(mel, axis=1)):
        results[f"MelBand_{i+1}"] = val

    # -------- PITCH CONTOUR --------
    f0c, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
    )
    f0c = f0c[~np.isnan(f0c)]
    results.update(summarize_feature(f0c, "PitchContour"))
    results["PitchContour_slope"] = np.polyfit(np.arange(len(f0c)), f0c, 1)[0] if len(f0c) > 1 else 0

    # -------- RHYTHM --------
    results.update(calculate_rhythm_features(y, sr))

    return results

# ===================== eGeMAPS =====================
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv01a,
    feature_level=opensmile.FeatureLevel.Functionals
)

def extract_egemaps_features(audio_path):
    return smile.process_file(audio_path).iloc[0].to_dict()

# ===================== BATCH PROCESSING INTO ONE CSV =====================
languages = ["Italian", "english"]
folders = ["HC", "PD"]
label_map = {"HC": 0, "PD": 1}

all_rows = []

for language in languages:
    for folder in folders:
        data_dir = os.path.join(DATA_ROOT, language, folder)

        print(f"\n--- Processing: {language}/{folder} ---")

        if not os.path.exists(data_dir):
            print(f"Skipping {language}/{folder}: Directory not found at {data_dir}")
            continue

        for fname in sorted(os.listdir(data_dir)):
            if not fname.lower().endswith(".wav"):
                continue

            file_path = os.path.join(data_dir, fname)
            print(f"Processing: {fname}")

            try:
                custom = extract_all_features(file_path)
                egemaps = extract_egemaps_features(file_path)
                features = {**custom, **egemaps}

                base_name = os.path.splitext(fname)[0]
                patient_id = base_name.split("-")[0]

                features["Patient_ID"] = patient_id
                features["Class"] = label_map[folder]
                features["File"] = fname
                features["Folder"] = folder
                features["Language"] = language

                all_rows.append(features)

                print(f"✔ Features extracted: {len(features)}")

            except Exception as e:
                print(f"❌ Failed on {fname}: {e}")

# ===================== SAVE FINAL CSV =====================
if len(all_rows) > 0:
    final_df = pd.DataFrame(all_rows)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ DONE — Combined CSV saved to:\n{OUTPUT_CSV}")
    print(f"Shape: {final_df.shape}")
else:
    print("\n❌ No data was processed. CSV not created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Processing: Italian/HC ---
Processing: HC1-1.wav
✔ Features extracted: 799
Processing: HC1-2.wav
✔ Features extracted: 799
Processing: HC10-1.wav
✔ Features extracted: 799
Processing: HC10-2.wav
✔ Features extracted: 799
Processing: HC11-1.wav
✔ Features extracted: 799
Processing: HC11-2.wav
✔ Features extracted: 799
Processing: HC12-1.wav
✔ Features extracted: 799
Processing: HC12-2.wav
✔ Features extracted: 799
Processing: HC13-1.wav
✔ Features extracted: 799
Processing: HC13-2.wav
✔ Features extracted: 799
Processing: HC14-1.wav
✔ Features extracted: 799
Processing: HC14-2.wav
✔ Features extracted: 799
Processing: HC15-1.wav
✔ Features extracted: 799
Processing: HC15-2.wav
✔ Features extracted: 799
Processing: HC16-1.wav
✔ Features extracted: 799
Processing: HC16-2.wav
✔ Features extracted: 799
Processing: HC17-1.wav
✔ Features extracted: 799
Processi

In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/outputs/parkinsons_audio_features_combined.csv")

print(df.shape)
df.head()

Mounted at /content/drive
(134, 799)


,Jitter_local,Jitter_local_abs,Jitter_rap,Jitter_ppq5,Jitter_ddp,Shimmer_local,Shimmer_local_dB,Shimmer_apq3,Shimmer_apq5,Shimmer_apq11,...,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp,Patient_ID,Class,File,Folder,Language
0,0.016313,0.000101,0.006418,0.007791,0.019253,0.120639,1.116089,0.047590,0.068529,0.131115,...,0.248310,0.190366,0.201053,0.281959,-24.468056,HC1,0,HC1-1.wav,HC,Italian
1,0.015619,0.000104,0.006890,0.008256,0.020670,0.102259,1.005739,0.032499,0.055372,0.115295,...,0.282424,0.207526,0.213763,0.257919,-27.610735,HC1,0,HC1-2.wav,HC,Italian
2,0.022460,0.000164,0.010799,0.011209,0.032396,0.105664,1.027141,0.043215,0.060274,0.109673,...,0.259444,0.244907,0.172000,0.198946,-25.467604,HC10,0,HC10-1.wav,HC,Italian
3,0.022797,0.000182,0.009554,0.010026,0.028662,0.105621,1.058577,0.039324,0.058338,0.112942,...,0.244819,0.196753,0.204524,0.225586,-24.692427,HC10,0,HC10-2.wav,HC,Italian
4,0.018983,0.000088,0.008349,0.009772,0.025047,0.107486,1.053217,0.039869,0.063825,0.116602,...,0.238714,0.213099,0.184762,0.220949,-28.035038,HC11,0,HC11-1.wav,HC,Italian


In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

directory_path = "/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/"

if os.path.exists(directory_path):
    print(f"The directory '{directory_path}' exists.")
else:
    print(f"The directory '{directory_path}' does not exist. Please ensure your Google Drive is mounted and the path is correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The directory '/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/' exists.


Cell 2 — Mount Drive, import libraries, load CSV

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import sys
if 'google.colab' in sys.modules:
    !pip install -U scikit-learn xgboost

import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_selection import RFECV, VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# ===================== PATHS =====================
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings"
CSV_PATH = os.path.join(PROJECT_DIR, "outputs", "parkinsons_audio_features_combined.csv")
OUT_DIR = os.path.join(PROJECT_DIR, "outputs", "rfecv_9_models_results")
os.makedirs(OUT_DIR, exist_ok=True)

# ===================== LOAD DATA =====================
df = pd.read_csv(CSV_PATH)

# If Patient_ID is missing, rebuild it from File names like PD1-1.wav -> PD1
if "Patient_ID" not in df.columns:
    if "File" not in df.columns:
        raise ValueError("CSV must contain either Patient_ID or File column.")
    df["Patient_ID"] = (
        df["File"]
        .astype(str)
        .str.replace(".wav", "", regex=False)
        .str.split("-")
        .str[0]
    )

print("Shape:", df.shape)
print("\nSample rows:")
display(df[["File", "Patient_ID", "Class"]].head())

print("\nUnique patients:", df["Patient_ID"].nunique())
print("Class counts:")
print(df["Class"].value_counts().sort_index())

# ===================== FEATURES / LABELS / GROUPS =====================
meta_cols = ["Patient_ID", "File", "Folder", "Class", "Language"]
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].astype(float).values
y = df["Class"].astype(int).values
groups = df["Patient_ID"].astype(str).values
feature_names = np.array(feature_cols)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (134, 799)

Sample rows:


,File,Patient_ID,Class
0,HC1-1.wav,HC1,0
1,HC1-2.wav,HC1,0
2,HC10-1.wav,HC10,0
3,HC10-2.wav,HC10,0
4,HC11-1.wav,HC11,0



Unique patients: 87
Class counts:
Class
0    64
1    70
Name: count, dtype: int64


Cell 3 — Verify that grouped folds keep PD1-1 and PD1-2 together

In [5]:
cv_check = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, test_idx) in enumerate(cv_check.split(X, y, groups), start=1):
    train_groups = set(groups[train_idx])
    test_groups = set(groups[test_idx])
    overlap = train_groups.intersection(test_groups)

    print(f"Fold {fold}")
    print("  Train patients:", len(train_groups))
    print("  Test patients :", len(test_groups))
    print("  Overlap       :", len(overlap))   # must be 0
    print("  Train class distribution:", np.bincount(y[train_idx]))
    print("  Test class distribution :", np.bincount(y[test_idx]))
    print("-" * 50)

Fold 1
  Train patients: 70
  Test patients : 17
  Overlap       : 0
  Train class distribution: [51 56]
  Test class distribution : [13 14]
--------------------------------------------------
Fold 2
  Train patients: 69
  Test patients : 18
  Overlap       : 0
  Train class distribution: [51 56]
  Test class distribution : [13 14]
--------------------------------------------------
Fold 3
  Train patients: 70
  Test patients : 17
  Overlap       : 0
  Train class distribution: [51 56]
  Test class distribution : [13 14]
--------------------------------------------------
Fold 4
  Train patients: 70
  Test patients : 17
  Overlap       : 0
  Train class distribution: [52 56]
  Test class distribution : [12 14]
--------------------------------------------------
Fold 5
  Train patients: 69
  Test patients : 18
  Overlap       : 0
  Train class distribution: [51 56]
  Test class distribution : [13 14]
--------------------------------------------------


Define the 9 RFECV models

In [6]:
MODELS = {
    "LogisticRegression": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "LinearSVM": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(
                class_weight="balanced",
                max_iter=20000,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "SGDClassifier": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                max_iter=3000,
                tol=1e-3,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "DecisionTree": {
        "estimator": DecisionTreeClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "RandomForest": {
        "estimator": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    },

    "ExtraTrees": {
        "estimator": ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    },

    "AdaBoost": {
        "estimator": AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "GradientBoosting": {
        "estimator": GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "XGBoost": {
        "estimator": XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "importance_getter": "auto"
    }
}

Helper functions for nested RFECV

In [7]:
def get_positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    else:
        return model.predict(X)


def evaluate_model_with_nested_rfecv(
    model_name,
    base_estimator,
    importance_getter,
    X,
    y,
    groups,
    feature_names,
    outer_splits=5,
    inner_splits=3,
    step=25,
    min_features=10,
):
    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fold_rows = []
    feature_counter = Counter()

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):
        print(f"\n[{model_name}] Fold {fold_idx}/{outer_splits}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_train, groups_test = groups[train_idx], groups[test_idx]

        # Remove constant features using TRAIN ONLY
        vt = VarianceThreshold(threshold=0.0)
        X_train_v = vt.fit_transform(X_train)
        X_test_v = vt.transform(X_test)
        kept_feature_names = feature_names[vt.get_support()]

        # Inner CV for RFECV
        inner_cv = StratifiedGroupKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        selector = RFECV(
            estimator=clone(base_estimator),
            step=step,
            cv=inner_cv,
            scoring="f1",
            min_features_to_select=min_features,
            n_jobs=-1,
            importance_getter=importance_getter
        )

        selector.fit(X_train_v, y_train, groups=groups_train)

        selected_features = kept_feature_names[selector.support_]
        for feat in selected_features:
            feature_counter[feat] += 1

        X_train_sel = selector.transform(X_train_v)
        X_test_sel = selector.transform(X_test_v)

        final_model = clone(base_estimator)
        final_model.fit(X_train_sel, y_train)

        y_pred = final_model.predict(X_test_sel)
        y_score = get_positive_scores(final_model, X_test_sel)

        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except ValueError:
            roc_auc = np.nan

        overlap = len(set(groups_train).intersection(set(groups_test)))

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_idx,
            "Train_Patients": len(np.unique(groups_train)),
            "Test_Patients": len(np.unique(groups_test)),
            "Patient_Overlap": overlap,
            "Selected_Features": int(selector.n_features_),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc
        })

        print("  Selected features:", selector.n_features_)
        print("  Patient overlap  :", overlap)
        print("  F1               :", round(f1_score(y_test, y_pred, zero_division=0), 4))
        print("  ROC_AUC          :", round(float(roc_auc), 4) if not np.isnan(roc_auc) else "nan")

    fold_df = pd.DataFrame(fold_rows)

    summary_row = {
        "Model": model_name,
        "Mean_Selected_Features": fold_df["Selected_Features"].mean(),
        "Accuracy_Mean": fold_df["Accuracy"].mean(),
        "Accuracy_SD": fold_df["Accuracy"].std(),
        "Precision_Mean": fold_df["Precision"].mean(),
        "Recall_Mean": fold_df["Recall"].mean(),
        "F1_Mean": fold_df["F1"].mean(),
        "F1_SD": fold_df["F1"].std(),
        "ROC_AUC_Mean": fold_df["ROC_AUC"].mean(),
        "ROC_AUC_SD": fold_df["ROC_AUC"].std(),
    }

    feature_freq_df = (
        pd.DataFrame(feature_counter.items(), columns=["Feature", "Selected_in_Folds"])
        .sort_values(["Selected_in_Folds", "Feature"], ascending=[False, True])
        .reset_index(drop=True)
    )

    return fold_df, summary_row, feature_freq_df

Run all 9 RFECV experiments and save outputs to Drive

In [8]:
summary_rows = []
all_fold_dfs = []

for model_name, cfg in MODELS.items():
    fold_df, summary_row, feature_freq_df = evaluate_model_with_nested_rfecv(
        model_name=model_name,
        base_estimator=cfg["estimator"],
        importance_getter=cfg["importance_getter"],
        X=X,
        y=y,
        groups=groups,
        feature_names=feature_names,
        outer_splits=5,
        inner_splits=3,      # keep 3 for speed; you can change to 5 later
        step=25,             # remove 25 features at a time; faster than step=1
        min_features=10
    )

    summary_rows.append(summary_row)
    all_fold_dfs.append(fold_df)

    # Save per-model outputs immediately
    fold_df.to_csv(os.path.join(OUT_DIR, f"{model_name}_fold_results.csv"), index=False)
    feature_freq_df.to_csv(
        os.path.join(OUT_DIR, f"{model_name}_selected_features_frequency.csv"),
        index=False
    )

    pd.DataFrame(summary_rows).to_csv(
        os.path.join(OUT_DIR, "rfecv_9_models_summary_progress.csv"),
        index=False
    )

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["F1_Mean", "ROC_AUC_Mean"],
    ascending=False
)

folds_df = pd.concat(all_fold_dfs, ignore_index=True)

summary_df.to_csv(os.path.join(OUT_DIR, "rfecv_9_models_summary.csv"), index=False)
folds_df.to_csv(os.path.join(OUT_DIR, "rfecv_9_models_all_folds.csv"), index=False)

print("\nFinal Summary:")
display(summary_df)
print("\nSaved to:", OUT_DIR)


[LogisticRegression] Fold 1/5
  Selected features: 257
  Patient overlap  : 0
  F1               : 0.88
  ROC_AUC          : 1.0

[LogisticRegression] Fold 2/5
  Selected features: 182
  Patient overlap  : 0
  F1               : 0.8966
  ROC_AUC          : 0.989

[LogisticRegression] Fold 3/5
  Selected features: 207
  Patient overlap  : 0
  F1               : 0.88
  ROC_AUC          : 0.978

[LogisticRegression] Fold 4/5
  Selected features: 157
  Patient overlap  : 0
  F1               : 0.9286
  ROC_AUC          : 0.9464

[LogisticRegression] Fold 5/5
  Selected features: 10
  Patient overlap  : 0
  F1               : 0.8485
  ROC_AUC          : 0.8736

[LinearSVM] Fold 1/5
  Selected features: 10
  Patient overlap  : 0
  F1               : 0.8966
  ROC_AUC          : 0.978

[LinearSVM] Fold 2/5
  Selected features: 57
  Patient overlap  : 0
  F1               : 0.963
  ROC_AUC          : 0.9835

[LinearSVM] Fold 3/5
  Selected features: 232
  Patient overlap  : 0
  F1             

,Model,Mean_Selected_Features,Accuracy_Mean,Accuracy_SD,Precision_Mean,Recall_Mean,F1_Mean,F1_SD,ROC_AUC_Mean,ROC_AUC_SD
5,ExtraTrees,347.0,0.954986,0.031687,0.971429,0.942857,0.956614,0.029731,0.989240,0.009982
4,RandomForest,547.0,0.939886,0.050829,0.946667,0.942857,0.943806,0.045710,0.982967,0.017201
6,AdaBoost,23.8,0.932479,0.049301,0.949321,0.928571,0.935631,0.045505,0.978755,0.021461
1,LinearSVM,102.6,0.917664,0.041389,0.944615,0.900000,0.918784,0.041916,0.971612,0.017992
2,SGDClassifier,237.0,0.917949,0.060785,0.929199,0.914286,0.918077,0.068100,0.961172,0.022136
0,LogisticRegression,162.6,0.880912,0.039805,0.906416,0.885714,0.886722,0.029157,0.957418,0.050935
8,XGBoost,247.6,0.873219,0.041896,0.888028,0.885714,0.881296,0.029669,0.971886,0.016161
3,DecisionTree,557.0,0.784330,0.083157,0.784836,0.842857,0.802135,0.073400,0.781044,0.083856
7,GradientBoosting,207.6,0.791453,0.054705,0.806068,0.800000,0.797949,0.058959,0.852106,0.060951



Saved to: /content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/outputs/rfecv_9_models_results
